In [1]:
import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import pandas as pd
import numpy as np
import json
import pickle
from tqdm import tqdm
from scipy import stats
# wilcoxon signed rank test: 1. ties and zeros. 2. median.
# paired t-test: 1. normal distribution assumption. 40 data points for each model x dataset combination.
# paired t-test more reliable.
import seaborn as sns
from matplotlib import font_manager
# Specify the path to the Calibri font file on your system
calibri_path = "/hpc/home/yx240/fonts/Arial.ttf"  # Example path, adjust as necessary
font_manager.fontManager.addfont(calibri_path)
calibri_path = "/hpc/home/yx240/fonts/Arial_Bold.ttf"  # Example path, adjust as necessary
font_manager.fontManager.addfont(calibri_path)
# Retrieve the pastel color palette
pastel_palette = sns.color_palette("pastel")

# Convert the colors to hexadecimal
pastel_colors_hex = pastel_palette.as_hex()

# Output the hex color codes
pastel_colors_hex

from matplotlib.colors import LinearSegmentedColormap
colors = [pastel_colors_hex[0], "#FFFFFF", pastel_colors_hex[1]]
# Create a custom colormap
cmap_name = 'pastel_blue_white_pastel_orange'
cm1 = LinearSegmentedColormap.from_list(cmap_name, colors)
colors = [pastel_colors_hex[1], "#FFFFFF", pastel_colors_hex[0]]
# Create a custom colormap
cmap_name = 'pastel_blue_white_pastel_orange'
cm2 = LinearSegmentedColormap.from_list(cmap_name, colors)

def get_rank_id(values, return_nan=False, ascending=False, test='wilcoxon'):
    valid_index = []
    if ascending:
        for i in range(len(values)):
            values[i] = - values[i]
    for i, v in enumerate(values):
        if v.__class__ == np.ndarray:
            valid_index.append(i)
    rank_id = [0] * len(valid_index)
    v_mean = [values[i].mean() for i in valid_index]
    while 0 in rank_id:
        current_rank_id = len(rank_id) - rank_id.count(0) + 1
        n_best = np.argmax(v_mean)
        assert rank_id[n_best] == 0
        rank_id[n_best] = current_rank_id
        v_mean[n_best] = - 100.
        for i in range(len(rank_id)):
            if i != n_best and rank_id[i] == 0:
                if np.array_equal(values[valid_index[i]], values[valid_index[n_best]]):
                    rank_id[i] = current_rank_id
                    v_mean[i] = - 100.
                    continue
                if test == 'wilcoxon':
                    v = stats.wilcoxon(values[valid_index[i]], values[valid_index[n_best]], alternative='less')
                elif test == 'paired t-test':
                    v = stats.ttest_rel(values[valid_index[i]], values[valid_index[n_best]], alternative='less')
                if v.pvalue > 0.05:
                    rank_id[i] = current_rank_id
                    v_mean[i] = - 100.
    if return_nan:
        return [rank_id[valid_index.index(i)] if i in valid_index else values[i] for i in range(len(values))]
    else:
        return rank_id
    
def format_model_names(models):
    return [m.replace('RBF-RDKit', 'RDKit').replace('DPK-Morgan', 'Morgan').
            replace('GPRv', r'GPR$\rm _{dbu}$').replace('GPRu', r'GPR$\rm _{pu}$') for m in models]

In [2]:
datasets_s = ['carcinogens_lagunin', 'skin', 'dili', 'hia_hou', 
              'bioavailability_ma', 'herg']
datasets_l = ['pgp_broccatelli', 'clintox', 'bace', 'bbbp', 'ames', 'CYP2C19_Veith']
datasets = datasets_s + datasets_l
datasets_name = ['Carcinogens', 'Skin', 'DILI', 'HIA', 'Bioavailability', 'hERG', 
                 'Pgp', 'ClinTox', 'BACE', 'BBBP', 'Ames', '2C19']
models_cv = ['RF-Morgan', 'RF-RDKit', 
          'GPR-DPK-Morgan', 'GPC-DPK-Morgan', 'SVC-DPK-Morgan', 
          'GPR-RBF-RDKit', 'GPC-RBF-RDKit', 'SVC-RBF-RDKit', 
          'GPR-MGK', 'GPC-MGK', 'SVC-MGK', 
          'GPR-MGK-DPK-Morgan', 'GPC-MGK-DPK-Morgan', 'SVC-MGK-DPK-Morgan', 
          'GPR-MGK-RBF-RDKit', 'GPC-MGK-RBF-RDKit', 'SVC-MGK-RBF-RDKit', 
          'MLP-Morgan', 'MLP-RDKit', 'D-MPNN', 'D-MPNN-Morgan', 'D-MPNN-RDKit']
models_deep = ['MLP-Morgan', 'MLP-RDKit', 'D-MPNN', 'D-MPNN-Morgan', 'D-MPNN-RDKit']
models_batch_inf = ['GPRu-DPK-Morgan', 'GPRu-RBF-RDKit', 'GPRu-MGK', 'GPRu-MGK-DPK-Morgan', 'GPRu-MGK-RBF-RDKit']
models_classical = ['RF-Morgan', 'RF-RDKit', 
          'GPRv-DPK-Morgan', 'GPC-DPK-Morgan', 'SVC-DPK-Morgan', 
          'GPRv-RBF-RDKit', 'GPC-RBF-RDKit', 'SVC-RBF-RDKit', ]
models_mgk = ['GPRv-MGK', 'GPC-MGK', 'SVC-MGK', 
          'GPRv-MGK-DPK-Morgan', 'GPC-MGK-DPK-Morgan', 'SVC-MGK-DPK-Morgan', 
          'GPRv-MGK-RBF-RDKit', 'GPC-MGK-RBF-RDKit', 'SVC-MGK-RBF-RDKit']
models_al = ['RF-Morgan', 'RF-RDKit', 
          'GPRu-DPK-Morgan', 'GPRv-DPK-Morgan', 'GPC-DPK-Morgan', 'SVC-DPK-Morgan', 
          'GPRu-RBF-RDKit', 'GPRv-RBF-RDKit', 'GPC-RBF-RDKit', 'SVC-RBF-RDKit', 
          'GPRu-MGK', 'GPRv-MGK', 'GPC-MGK', 'SVC-MGK', 
          'GPRu-MGK-DPK-Morgan', 'GPRv-MGK-DPK-Morgan', 'GPC-MGK-DPK-Morgan', 'SVC-MGK-DPK-Morgan', 
          'GPRu-MGK-RBF-RDKit', 'GPRv-MGK-RBF-RDKit', 'GPC-MGK-RBF-RDKit', 'SVC-MGK-RBF-RDKit', 
          'MLP-Morgan', 'MLP-RDKit', 'D-MPNN', 'D-MPNN-Morgan', 'D-MPNN-RDKit']
models_morgan = ['RF-Morgan', 'GPR-DPK-Morgan', 
                 'GPR-MGK-DPK-Morgan', 'MLP-Morgan', 'D-MPNN-Morgan']
models_rdkit = ['RF-RDKit', 'GPR-RBF-RDKit', 
                'GPR-MGK-RBF-RDKit', 'MLP-RDKit', 'D-MPNN-RDKit']
metrics = ['roc-auc', 'mcc', 'accuracy', 'precision', 'recall', 'f1_score']
metrics_label = ['ROC-AUC', 'MCC', 'ACC', 'Precision', 'Recall', 'F1 score']
splits = ['random', 'scaffold_random']
ps = [25, 50, 75, 100]
df_cv = pd.read_csv('data/cv.csv')
df_al = pd.read_csv('data/AL.csv')
df_yl = pd.read_csv('data/YoL.csv')
# models * datasets * splits * random seeds * metrics

n_datasets = [280, 404, 475, 578, 640, 655, 1218, 1478, 1513, 2039, 7278, 12665]
n_strides = [2, 2, 3, 4, 4, 4] + [5] * 6
training_sizes = [list(range(n_strides[i], min(500, int(n_datasets[i] / 2) + 1), n_strides[i])) for i, n in enumerate(n_datasets)]
for i, n in enumerate(n_datasets):
    if int(n_datasets[i] / 2) + 1 > 500:
        training_sizes[i].append(500)
    elif training_sizes[i][-1] != int((n_datasets[i] + 1) / 2):
        training_sizes[i].append(int((n_datasets[i] + 1) / 2))
training_sizes_batch10 = [list(range(2, min(500, int(n_datasets[i] / 2) + 1), 10)) for i, n in enumerate(n_datasets)]

In [3]:
models_top5 = ['GPRv-MGK-RBF-RDKit', 'RF-RDKit', 'MLP-RDKit', 'GPRv-RBF-RDKit', 'D-MPNN-RDKit']
for i in range(4):
    v1 = df_al[(df_al['learning_type'] == 'active')&(df_al['model'] == models_top5[i])]['aulc']
    v2 = df_al[(df_al['learning_type'] == 'active')&(df_al['model'] == models_top5[i+1])]['aulc']
    print(f"{models_top5[i]} is significantly better than {models_top5[i+1]} with pvalue {stats.wilcoxon(v1, v2, alternative='greater').pvalue}")

GPRv-MGK-RBF-RDKit is significantly better than RF-RDKit with pvalue 2.4499860157031044e-08
RF-RDKit is significantly better than MLP-RDKit with pvalue 1.2678102236071292e-09
MLP-RDKit is significantly better than GPRv-RBF-RDKit with pvalue 0.0030958481355768575
GPRv-RBF-RDKit is significantly better than D-MPNN-RDKit with pvalue 0.0006763109111740689


In [4]:
models = (models_classical + models_mgk + models_deep + models_batch_inf)
values = np.zeros((len(models_al), len(models_al)))
df1 = df_al[df_al['learning_type'] == 'active']
N_count = 0
for dataset in datasets:
    # print(dataset)
    df2 = df1[df1['dataset'] == dataset]
    for split in splits:
        df3 = df2[df2['split'] == split]
        for metric in metrics:
            df4 = df3[df3['metric'] == metric]
            for seed in range(20):
                df5 = df4[df4['seed'] == seed]
                idx = [df5.model.tolist().index(m) for m in models]
                df5 = df5.iloc[idx]
                LC = np.array(df5['lc'].apply(lambda x: json.loads(x)).tolist())
                for i in range(LC.shape[1]):
                    for j, k in enumerate(np.argsort(-LC[:, i])):
                        values[j][k] += 1  # rank j, k-th model.
                    N_count += 1
                assert len(df5) == 27

mean_rank = 0.
for i, v in enumerate(values):
    mean_rank += v * (i + 1)
mean_rank /= N_count
model_rank_idx = np.argsort(mean_rank)

models = np.array(models_classical + models_mgk + models_deep + models_batch_inf)
models_rank = np.array(format_model_names(models))[model_rank_idx]
models_name = [m + ' (%.1f)' % mean_rank[model_rank_idx][i] for i, m in enumerate(models_rank)]
values = np.zeros((len(models_al), len(models_al)))
df1 = df_al[df_al['learning_type'] == 'active']
dfs = []
for dataset in datasets:
    # print(dataset)
    df2 = df1[df1['dataset'] == dataset]
    for split in splits:
        df3 = df2[df2['split'] == split]
        for metric in metrics:
            df4 = df3[df3['metric'] == metric]
            for seed in range(20):
                df5 = df4[df4['seed'] == seed]
                idx = [df5.model.tolist().index(m) for m in models]
                df5 = df5.iloc[idx]
                LC = np.array(df5['lc'].apply(lambda x: json.loads(x)).tolist())
                for i in range(LC.shape[1]):
                    df_ = pd.DataFrame({'model': models_name, 'rank': (np.argsort(np.argsort(-LC[:, i])) + 1)[model_rank_idx]})
                    dfs.append(df_)
                assert len(df5) == 27
df = pd.concat(dfs)

In [5]:
# top1 probability
df[df['rank'] == 1].value_counts() / len(df[df['rank'] == 1])

model                              rank
RF-RDKit (7.7)                     1       0.144114
GPR$\rm _{dbu}$-MGK-RDKit (6.7)    1       0.141732
D-MPNN-RDKit (8.7)                 1       0.094299
MLP-RDKit (8.1)                    1       0.075426
GPR$\rm _{dbu}$-RDKit (9.0)        1       0.065915
D-MPNN (13.2)                      1       0.052590
GPR$\rm _{dbu}$-MGK (11.7)         1       0.050871
RF-Morgan (13.1)                   1       0.048132
SVC-MGK-RDKit (11.1)               1       0.047932
GPC-MGK-RDKit (12.2)               1       0.042940
GPR$\rm _{pu}$-RDKit (13.3)        1       0.033752
GPR$\rm _{pu}$-MGK-RDKit (13.5)    1       0.031918
SVC-RDKit (13.6)                   1       0.029896
GPC-RDKit (14.6)                   1       0.029547
SVC-MGK-Morgan (14.5)              1       0.014771
GPC-MGK-Morgan (12.5)              1       0.012653
GPR$\rm _{dbu}$-MGK-Morgan (14.8)  1       0.012588
MLP-Morgan (16.8)                  1       0.011169
D-MPNN-Morgan (15.7)    

In [6]:
# RDKit VS Morgan. Active Learning Performance.
for mr, mm in [('RF-RDKit', 'RF-Morgan'), 
               ('GPRv-RBF-RDKit', 'GPRv-DPK-Morgan'), ('GPRv-MGK-RBF-RDKit', 'GPRv-MGK-DPK-Morgan'), 
               ('GPRu-RBF-RDKit', 'GPRu-DPK-Morgan'), ('GPRu-MGK-RBF-RDKit', 'GPRv-MGK-DPK-Morgan'), 
               ('MLP-RDKit', 'MLP-Morgan'), ('D-MPNN-RDKit', 'D-MPNN-Morgan'), 
               ('SVC-RBF-RDKit', 'SVC-DPK-Morgan'), ('SVC-MGK-RBF-RDKit', 'SVC-MGK-DPK-Morgan'), 
              ]:
    v1 = df_al[(df_al['learning_type'] == 'active')&(df_al['model'] == mr)]['aulc']
    v2 = df_al[(df_al['learning_type'] == 'active')&(df_al['model'] == mm)]['aulc']
    print(f"{mr} > {mm} with pvalue {stats.wilcoxon(v1, v2, alternative='greater').pvalue}")
for mr, mm in [('GPC-RBF-RDKit', 'GPC-DPK-Morgan'), ('GPC-MGK-RBF-RDKit', 'GPC-MGK-DPK-Morgan')]:
    v1 = df_al[(df_al['learning_type'] == 'active')&(df_al['model'] == mr)]['aulc']
    v2 = df_al[(df_al['learning_type'] == 'active')&(df_al['model'] == mm)]['aulc']
    print(f"{mr} < {mm} with pvalue {stats.wilcoxon(v1, v2, alternative='less').pvalue}")

RF-RDKit > RF-Morgan with pvalue 1.031078702254028e-246
GPRv-RBF-RDKit > GPRv-DPK-Morgan with pvalue 0.0
GPRv-MGK-RBF-RDKit > GPRv-MGK-DPK-Morgan with pvalue 0.0
GPRu-RBF-RDKit > GPRu-DPK-Morgan with pvalue 0.0
GPRu-MGK-RBF-RDKit > GPRv-MGK-DPK-Morgan with pvalue 3.81551934657252e-14
MLP-RDKit > MLP-Morgan with pvalue 2.32436387602818e-309
D-MPNN-RDKit > D-MPNN-Morgan with pvalue 1.9950444016513108e-291
SVC-RBF-RDKit > SVC-DPK-Morgan with pvalue 1.186919628546151e-74
SVC-MGK-RBF-RDKit > SVC-MGK-DPK-Morgan with pvalue 1.7886695049527712e-65
GPC-RBF-RDKit < GPC-DPK-Morgan with pvalue 6.906696066141357e-07
GPC-MGK-RBF-RDKit < GPC-MGK-DPK-Morgan with pvalue 0.0019265576533304516


In [7]:
models_top5 = ['GPR-MGK-RBF-RDKit', 'RF-RDKit', 'D-MPNN-RDKit', 'MLP-RDKit', 'GPR-RBF-RDKit']
for i in range(4):
    v1 = []
    for metric in metrics:
        v1 += df_cv[(df_cv['model'] == models_top5[i])][metric].tolist()
    v2 = []
    for metric in metrics:
        v2 += df_cv[(df_cv['model'] == models_top5[i+1])][metric].tolist()
    print(f"{models_top5[i]} is significantly better than {models_top5[i+1]} with pvalue {stats.wilcoxon(v1, v2, alternative='greater').pvalue}")

GPR-MGK-RBF-RDKit is significantly better than RF-RDKit with pvalue 7.957692895163898e-13
RF-RDKit is significantly better than D-MPNN-RDKit with pvalue 2.701612827626651e-09
D-MPNN-RDKit is significantly better than MLP-RDKit with pvalue 0.07754157181717507
MLP-RDKit is significantly better than GPR-RBF-RDKit with pvalue 0.0005213715610532994


In [8]:
# RDKit VS Morgan. Prediction Performance
for mr, mm in [('RF-RDKit', 'RF-Morgan'), 
               ('GPR-RBF-RDKit', 'GPR-DPK-Morgan'), ('GPR-MGK-RBF-RDKit', 'GPR-MGK-DPK-Morgan'), 
               ('MLP-RDKit', 'MLP-Morgan'), ('D-MPNN-RDKit', 'D-MPNN-Morgan'), 
               ('SVC-RBF-RDKit', 'SVC-DPK-Morgan'), ('SVC-MGK-RBF-RDKit', 'SVC-MGK-DPK-Morgan'), 
              ]:
    v1 = []
    for metric in metrics:
        v1 += df_cv[(df_cv['model'] == mr)][metric].tolist()
    v2 = []
    for metric in metrics:
        v2 += df_cv[(df_cv['model'] == mm)][metric].tolist()
    print(f"{mr} > {mm} with pvalue {stats.wilcoxon(v1, v2, alternative='greater').pvalue}")
for mr, mm in [('GPC-RBF-RDKit', 'GPC-DPK-Morgan'), ('GPC-MGK-RBF-RDKit', 'GPC-MGK-DPK-Morgan')]:
    v1 = []
    for metric in metrics:
        v1 += df_cv[(df_cv['model'] == mr)][metric].tolist()
    v2 = []
    for metric in metrics:
        v2 += df_cv[(df_cv['model'] == mm)][metric].tolist()
    print(f"{mr} < {mm} with pvalue {stats.wilcoxon(v1, v2, alternative='less').pvalue}")

RF-RDKit > RF-Morgan with pvalue 5.587172781229788e-216
GPR-RBF-RDKit > GPR-DPK-Morgan with pvalue 0.0
GPR-MGK-RBF-RDKit > GPR-MGK-DPK-Morgan with pvalue 0.0
MLP-RDKit > MLP-Morgan with pvalue 0.0
D-MPNN-RDKit > D-MPNN-Morgan with pvalue 1.3660862798441992e-293
SVC-RBF-RDKit > SVC-DPK-Morgan with pvalue 9.31503545349653e-306
SVC-MGK-RBF-RDKit > SVC-MGK-DPK-Morgan with pvalue 9.456225543330917e-213
GPC-RBF-RDKit < GPC-DPK-Morgan with pvalue 4.8446004832094755e-88
GPC-MGK-RBF-RDKit < GPC-MGK-DPK-Morgan with pvalue 6.202323118988547e-85


In [9]:
# yoked deep learning
for smodel in ['MLP-RDKit', 'D-MPNN-RDKit']:
    v2 = df_al[(df_al['learning_type'] == 'active')&(df_al['model'] == smodel)&(df_al.metric == 'roc-auc')]['aulc']
    for tmodel in ['RF-RDKit', 'GPRv-RBF-RDKit', 'GPRv-MGK-RBF-RDKit']:
        v1 = df_yl[(df_yl.model_selector == tmodel)&(df_yl.model_evaluator == smodel)&(df_yl.metric == 'roc-auc')]['aulc']
        print(f"{tmodel} yoked {smodel} is significantly better than {smodel} active learning with pvalue {stats.wilcoxon(v1, v2, alternative='greater').pvalue}")

RF-RDKit yoked MLP-RDKit is significantly better than MLP-RDKit active learning with pvalue 0.00017181782439679746
GPRv-RBF-RDKit yoked MLP-RDKit is significantly better than MLP-RDKit active learning with pvalue 1.8287255790146097e-10
GPRv-MGK-RBF-RDKit yoked MLP-RDKit is significantly better than MLP-RDKit active learning with pvalue 3.0357647373620453e-06
RF-RDKit yoked D-MPNN-RDKit is significantly better than D-MPNN-RDKit active learning with pvalue 9.515919083795582e-10
GPRv-RBF-RDKit yoked D-MPNN-RDKit is significantly better than D-MPNN-RDKit active learning with pvalue 9.84981631272393e-17
GPRv-MGK-RBF-RDKit yoked D-MPNN-RDKit is significantly better than D-MPNN-RDKit active learning with pvalue 1.7341489102011777e-16
